# WeatherGuard Mission 20 — Real Dataset GPU Training & Evaluation Harness
**Platform**: Kaggle NVIDIA Tesla T4 GPU Runtime (16 GB VRAM)
**Dataset**: `allfaz/weatherguard-v2-corpus` (1,000 Curated Frames)
**Splits**: Train (667 frames), Val (173 frames), Locked Test (160 frames, 0 augmented)
**Repository**: https://github.com/mohammad-sayem-uddin/weatherguard
**Committer**: Mohammad Sayem Uddin <sayemwork41@gmail.com>


In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


In [ ]:
!pip install -q ultralytics pyyaml pandas opencv-python pillow


In [ ]:
import os
from pathlib import Path
dataset_path = Path('/kaggle/input/weatherguard-v2-corpus')
print('Dataset directory exists:', dataset_path.exists())
if dataset_path.exists():
    for split in ['train', 'val', 'test']:
        img_cnt = len(list((dataset_path / split / 'images').glob('*.*')))
        lbl_cnt = len(list((dataset_path / split / 'labels').glob('*.txt')))
        print(f'  {split}: {img_cnt} images, {lbl_cnt} labels')


In [ ]:
from pathlib import Path
runner_content = '#!/usr/bin/env python3\n"""WeatherGuard — Reusable Kaggle GPU Training & Evaluation Runner.\n\nTarget Environment: Kaggle Cloud, NVIDIA Tesla T4 GPU (16 GB VRAM).\nCapabilities:\n  - CUDA GPU device validation and VRAM profiling.\n  - Reproducibility seeding (Python, NumPy, PyTorch, CUDA deterministic).\n  - Fine-tunes YOLOv8n on WeatherGuard V2 train split (667 frames) with AdamW, AMP.\n  - Checkpoint selection strictly on validation split (173 frames).\n  - Frozen independent evaluation on locked test split (160 frames, 0 augmented).\n  - Metrics: mAP@0.5, mAP@0.5:0.95, per-class AP, 4 scale categories, and 4 weather domains.\n  - Zero-shot baseline vs Fine-tuned comparison.\n  - Exports artifacts to /kaggle/working/artifacts/.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport glob\nimport json\nimport logging\nimport os\nimport random\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport cv2\nimport numpy as np\nimport torch\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("WeatherGuard-KaggleGPU")\n\nSEED = 42\n\nCANONICAL_CLASSES = {\n    0: "truck",\n    1: "person",\n    2: "bicycle",\n    3: "car",\n    4: "motorcycle",\n    5: "bus",\n}\n\n# COCO to WeatherGuard canonical class ID mapping\n# COCO: person=0, bicycle=1, car=2, motorcycle=3, bus=5, truck=7\nCOCO_TO_WG_CLASS_ID = {\n    0: 1,  # person -> 1\n    1: 2,  # bicycle -> 2\n    2: 3,  # car -> 3\n    3: 4,  # motorcycle -> 4\n    5: 5,  # bus -> 5\n    7: 0,  # truck -> 0\n}\n\n\ndef set_reproducibility(seed: int = SEED) -> None:\n    """Sets random seeds across all libraries for deterministic execution."""\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n    logger.info("Deterministic seed %d initialized.", seed)\n\n\ndef audit_gpu_environment() -> Dict[str, Any]:\n    """Inspects and returns the hardware runtime profile."""\n    is_cuda = torch.cuda.is_available()\n    device_name = torch.cuda.get_device_name(0) if is_cuda else "CPU"\n    total_mem_gb = (\n        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if is_cuda else 0.0\n    )\n\n    profile = {\n        "platform": "Kaggle Cloud",\n        "cuda_available": is_cuda,\n        "device_name": device_name,\n        "gpu_memory_gb": round(total_mem_gb, 2),\n        "pytorch_version": torch.__version__,\n        "cuda_version": torch.version.cuda if is_cuda else None,\n        "is_nvidia_t4": "T4" in device_name,\n        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n    }\n    logger.info("GPU Audit: %s (%.2f GB VRAM, CUDA %s)",\n                device_name, total_mem_gb, profile["cuda_version"])\n    return profile\n\n\ndef get_scale_bucket(width: float, height: float) -> str:\n    """Categorizes box into canonical scale buckets based on max dimension in pixels."""\n    max_d = max(width, height)\n    if max_d < 16.0:\n        return "ultra_small"\n    if max_d < 32.0:\n        return "small"\n    if max_d < 96.0:\n        return "medium"\n    return "large"\n\n\ndef get_weather_domain(filename: str) -> str:\n    """Extracts adverse weather domain from filename."""\n    name = Path(filename).name.lower()\n    if "b24d283f" in name or "rain" in name:\n        return "rain"\n    if "snow" in name:\n        return "snow"\n    if "b255cd6c" in name or "night" in name:\n        return "night"\n    if "dense_fog" in name or "fog" in name:\n        return "dense_fog"\n    return "other"\n\n\ndef compute_box_iou(boxA: List[float], boxB: List[float]) -> float:\n    """Computes Intersection over Union for [xmin, ymin, xmax, ymax]."""\n    xA = max(boxA[0], boxB[0])\n    yA = max(boxA[1], boxB[1])\n    xB = min(boxA[2], boxB[2])\n    yB = min(boxA[3], boxB[3])\n\n    inter = max(0.0, xB - xA) * max(0.0, yB - yA)\n    areaA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])\n    areaB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])\n    union = areaA + areaB - inter\n    if union <= 0.0:\n        return 0.0\n    return float(inter / union)\n\n\ndef evaluate_fine_grained(\n    model: Any,\n    test_img_dir: Path,\n    test_lbl_dir: Path,\n    is_zero_shot: bool = False,\n    conf_threshold: float = 0.25,\n    iou_match_threshold: float = 0.5,\n) -> Dict[str, Any]:\n    """Evaluates detections against ground truth split by scale and weather condition."""\n    scale_stats = {\n        "ultra_small": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0},\n        "small": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0},\n        "medium": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0},\n        "large": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0},\n    }\n    weather_stats = {\n        "rain": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0, "images": 0},\n        "snow": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0, "images": 0},\n        "night": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0, "images": 0},\n        "dense_fog": {"tp": 0, "fp": 0, "fn": 0, "total_gt": 0, "images": 0},\n    }\n\n    img_files = sorted([p for p in test_img_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")])\n    logger.info("Running fine-grained evaluation on %d test images (zero_shot=%s)...", len(img_files), is_zero_shot)\n\n    for img_p in img_files:\n        lbl_p = test_lbl_dir / f"{img_p.stem}.txt"\n        img = cv2.imread(str(img_p))\n        if img is None:\n            continue\n        h, w = img.shape[:2]\n\n        weather = get_weather_domain(img_p.name)\n        if weather in weather_stats:\n            weather_stats[weather]["images"] += 1\n\n        # Read GT\n        gt_boxes = []\n        if lbl_p.exists():\n            for line in lbl_p.read_text().strip().splitlines():\n                parts = line.strip().split()\n                if len(parts) >= 5:\n                    cls_id = int(parts[0])\n                    cx, cy, bw, bh = map(float, parts[1:5])\n                    xmin = (cx - bw / 2.0) * w\n                    ymin = (cy - bh / 2.0) * h\n                    xmax = (cx + bw / 2.0) * w\n                    ymax = (cy + bh / 2.0) * h\n                    pw = bw * w\n                    ph = bh * h\n                    scale = get_scale_bucket(pw, ph)\n                    gt_boxes.append({\n                        "cls_id": cls_id,\n                        "box": [xmin, ymin, xmax, ymax],\n                        "scale": scale,\n                        "matched": False,\n                    })\n\n        # Count GT totals\n        for gt in gt_boxes:\n            scale_stats[gt["scale"]]["total_gt"] += 1\n            if weather in weather_stats:\n                weather_stats[weather]["total_gt"] += 1\n\n        # Run inference\n        results = model.predict(source=str(img_p), conf=conf_threshold, iou=0.45, verbose=False)[0]\n        pred_boxes = []\n        if results.boxes is not None and len(results.boxes) > 0:\n            for b in results.boxes:\n                c = int(b.cls[0].item())\n                if is_zero_shot:\n                    if c not in COCO_TO_WG_CLASS_ID:\n                        continue\n                    c = COCO_TO_WG_CLASS_ID[c]\n                xyxy = b.xyxy[0].tolist()\n                pw = xyxy[2] - xyxy[0]\n                ph = xyxy[3] - xyxy[1]\n                scale = get_scale_bucket(pw, ph)\n                pred_boxes.append({\n                    "cls_id": c,\n                    "box": xyxy,\n                    "scale": scale,\n                    "matched": False,\n                })\n\n        # Match preds to GT (Greedy IoU)\n        for pred in pred_boxes:\n            best_iou = 0.0\n            best_gt_idx = -1\n            for idx, gt in enumerate(gt_boxes):\n                if not gt["matched"] and gt["cls_id"] == pred["cls_id"]:\n                    iou = compute_box_iou(pred["box"], gt["box"])\n                    if iou > best_iou:\n                        best_iou = iou\n                        best_gt_idx = idx\n\n            if best_iou >= iou_match_threshold and best_gt_idx >= 0:\n                gt_boxes[best_gt_idx]["matched"] = True\n                pred["matched"] = True\n                gt_scale = gt_boxes[best_gt_idx]["scale"]\n                scale_stats[gt_scale]["tp"] += 1\n                if weather in weather_stats:\n                    weather_stats[weather]["tp"] += 1\n            else:\n                scale_stats[pred["scale"]]["fp"] += 1\n                if weather in weather_stats:\n                    weather_stats[weather]["fp"] += 1\n\n        # Unmatched GT are FN\n        for gt in gt_boxes:\n            if not gt["matched"]:\n                scale_stats[gt["scale"]]["fn"] += 1\n                if weather in weather_stats:\n                    weather_stats[weather]["fn"] += 1\n\n    # Compute Precision, Recall, F1\n    scale_summary = {}\n    for s, data in scale_stats.items():\n        tp = data["tp"]\n        fp = data["fp"]\n        fn = data["fn"]\n        p = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0\n        r = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0\n        f1 = round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0\n        scale_summary[s] = {\n            "total_gt": data["total_gt"],\n            "tp": tp,\n            "fp": fp,\n            "fn": fn,\n            "precision": p,\n            "recall": r,\n            "f1": f1,\n        }\n\n    weather_summary = {}\n    for w_name, data in weather_stats.items():\n        tp = data["tp"]\n        fp = data["fp"]\n        fn = data["fn"]\n        p = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0\n        r = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0\n        f1 = round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0\n        weather_summary[w_name] = {\n            "images": data["images"],\n            "total_gt": data["total_gt"],\n            "tp": tp,\n            "fp": fp,\n            "fn": fn,\n            "precision": p,\n            "recall": r,\n            "f1": f1,\n        }\n\n    return {\n        "scale_metrics": scale_summary,\n        "weather_metrics": weather_summary,\n    }\n\n\ndef run_full_vision_benchmark(\n    data_yaml: str = "/kaggle/input/weatherguard-v2-corpus/data.yaml",\n    weights_path: str = "yolov8n.pt",\n    epochs: int = 15,\n    imgsz: int = 640,\n    batch_size: int = 16,\n    output_dir: str = "/kaggle/working/runs/train",\n    experiment_id: str = "weatherguard_t4_real",\n) -> Dict[str, Any]:\n    """Orchestrates end-to-end training and evaluation on Kaggle GPU."""\n    from ultralytics import YOLO\n\n    set_reproducibility(SEED)\n    gpu_profile = audit_gpu_environment()\n\n    data_dir = Path(data_yaml).parent\n    test_img_dir = data_dir / "test" / "images"\n    test_lbl_dir = data_dir / "test" / "labels"\n\n    # Step 1: Zero-shot baseline evaluation on test set\n    logger.info("--- Step 1: Evaluating Pretrained YOLOv8n (Zero-Shot Baseline) ---")\n    base_model = YOLO("yolov8n.pt")\n    zero_shot_fine_grained = evaluate_fine_grained(\n        base_model, test_img_dir, test_lbl_dir, is_zero_shot=True, conf_threshold=0.25\n    )\n\n    # Step 2: Fine-Tuning YOLOv8n on Train split (667 frames)\n    logger.info("--- Step 2: Fine-Tuning YOLOv8n on WeatherGuard V2 Train Split ---")\n    device = 0 if torch.cuda.is_available() else "cpu"\n    train_model = YOLO("yolov8n.pt")\n\n    start_train_time = time.perf_counter()\n    train_model.train(\n        data=data_yaml,\n        epochs=epochs,\n        imgsz=imgsz,\n        batch=batch_size,\n        device=device,\n        project=output_dir,\n        name=experiment_id,\n        seed=SEED,\n        deterministic=True,\n        amp=torch.cuda.is_available(),\n        optimizer="AdamW",\n        lr0=0.001,\n        lrf=0.01,\n        weight_decay=0.0005,\n        save=True,\n        val=True,\n        plots=True,\n        workers=2,\n    )\n    training_time_s = time.perf_counter() - start_train_time\n    logger.info("Training finished in %.2f seconds.", training_time_s)\n\n    best_ckpt = Path(output_dir) / experiment_id / "weights" / "best.pt"\n    if not best_ckpt.exists():\n        raise FileNotFoundError(f"Best checkpoint not found at {best_ckpt}")\n\n    # Step 3: Evaluate fine-tuned best model on Validation split (model selection check)\n    logger.info("--- Step 3: Validating Best Checkpoint on Validation Split ---")\n    val_eval_model = YOLO(str(best_ckpt))\n    val_results = val_eval_model.val(data=data_yaml, split="val", device=device, plots=False)\n    val_metrics = {\n        "precision": round(float(val_results.results_dict.get("metrics/precision(B)", 0.0)), 4),\n        "recall": round(float(val_results.results_dict.get("metrics/recall(B)", 0.0)), 4),\n        "mAP50": round(float(val_results.results_dict.get("metrics/mAP50(B)", 0.0)), 4),\n        "mAP50_95": round(float(val_results.results_dict.get("metrics/mAP50-95(B)", 0.0)), 4),\n    }\n\n    # Step 4: Frozen Independent Evaluation on Locked Test split (160 frames, 0 augmented)\n    logger.info("--- Step 4: Independent Evaluation on Locked Test Split ---")\n    test_results = val_eval_model.val(data=data_yaml, split="test", device=device, plots=True)\n    test_metrics = {\n        "precision": round(float(test_results.results_dict.get("metrics/precision(B)", 0.0)), 4),\n        "recall": round(float(test_results.results_dict.get("metrics/recall(B)", 0.0)), 4),\n        "mAP50": round(float(test_results.results_dict.get("metrics/mAP50(B)", 0.0)), 4),\n        "mAP50_95": round(float(test_results.results_dict.get("metrics/mAP50-95(B)", 0.0)), 4),\n    }\n\n    # Per-class AP on test split\n    per_class_test = {}\n    if hasattr(test_results.box, "ap50") and test_results.box.ap50 is not None:\n        ap50_list = test_results.box.ap50\n        ap_list = test_results.box.maps if hasattr(test_results.box, "maps") else [0.0] * len(ap50_list)\n        for idx, name in CANONICAL_CLASSES.items():\n            if idx < len(ap50_list):\n                per_class_test[name] = {\n                    "mAP50": round(float(ap50_list[idx]), 4),\n                    "mAP50_95": round(float(ap_list[idx]), 4),\n                }\n\n    # Fine-grained scale and weather evaluation on locked test split\n    ft_fine_grained = evaluate_fine_grained(\n        val_eval_model, test_img_dir, test_lbl_dir, is_zero_shot=False, conf_threshold=0.25\n    )\n\n    # Export artifacts\n    artifacts_dir = Path("/kaggle/working/artifacts/vision")\n    artifacts_dir.mkdir(parents=True, exist_ok=True)\n\n    # Copy key files\n    for item in ["best.pt", "last.pt"]:\n        src_f = Path(output_dir) / experiment_id / "weights" / item\n        if src_f.exists():\n            shutil.copy2(src_f, artifacts_dir / item)\n\n    for item in ["results.csv", "results.png", "confusion_matrix.png"]:\n        src_f = Path(output_dir) / experiment_id / item\n        if src_f.exists():\n            shutil.copy2(src_f, artifacts_dir / item)\n\n    benchmark_summary = {\n        "experiment_id": experiment_id,\n        "dataset_yaml": data_yaml,\n        "gpu_profile": gpu_profile,\n        "training": {\n            "epochs": epochs,\n            "imgsz": imgsz,\n            "batch_size": batch_size,\n            "optimizer": "AdamW",\n            "amp": True,\n            "training_time_seconds": round(training_time_s, 2),\n            "best_checkpoint": str(best_ckpt),\n        },\n        "validation_split_metrics": val_metrics,\n        "locked_test_split_metrics": {\n            "overall": test_metrics,\n            "per_class": per_class_test,\n            "scale_breakdown": ft_fine_grained["scale_metrics"],\n            "weather_breakdown": ft_fine_grained["weather_metrics"],\n        },\n        "zero_shot_baseline_comparison": {\n            "scale_breakdown": zero_shot_fine_grained["scale_metrics"],\n            "weather_breakdown": zero_shot_fine_grained["weather_metrics"],\n        },\n    }\n\n    out_json = artifacts_dir / "mission20_vision_gpu_results.json"\n    out_json.write_text(json.dumps(benchmark_summary, indent=2))\n    logger.info("Saved complete vision benchmark results to %s", out_json)\n\n    return benchmark_summary\n\n\nif __name__ == "__main__":\n    profile = audit_gpu_environment()\n    print(json.dumps(profile, indent=2))\n'
Path('/kaggle/working/weatherguard_gpu_runner.py').write_text(runner_content)
print('Runner written to /kaggle/working/weatherguard_gpu_runner.py')


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
from weatherguard_gpu_runner import run_full_vision_benchmark

benchmark_results = run_full_vision_benchmark(
    data_yaml='/kaggle/input/weatherguard-v2-corpus/data.yaml',
    weights_path='yolov8n.pt',
    epochs=15,
    imgsz=640,
    batch_size=16,
    output_dir='/kaggle/working/runs/train',
    experiment_id='weatherguard_t4_real',
)

print('--- BENCHMARK RESULTS SUMMARY ---')
print('Validation Split Metrics:', benchmark_results['validation_split_metrics'])
print('Locked Test Overall Metrics:', benchmark_results['locked_test_split_metrics']['overall'])
print('Locked Test Per-Class:', benchmark_results['locked_test_split_metrics']['per_class'])
print('Locked Test Scale Breakdown:', benchmark_results['locked_test_split_metrics']['scale_breakdown'])
print('Locked Test Weather Breakdown:', benchmark_results['locked_test_split_metrics']['weather_breakdown'])
